# Skill Extraction - v1 Gazetteer

The EDA notebooks established that we can't rely on structured skill fields: `skills_desc` is 98% empty, and `job_skills.csv` only has 35 broad functional categories ("Engineering", "Sales") rather than real skills. This notebook extracts actual skills directly from the free-text `description` field.

**Approach: gazetteer / dictionary matching.** We match a curated list of ~90 known skill terms (programming languages, tools, platforms, and common business/soft skills) against posting text, case-insensitively.

**Implementation note:** the first version of this used a combined regex (all terms as one alternation). That turned out to be too slow at this scale - matching ~90 alternatives against every character position across 123,849 documents is inherently expensive for a backtracking regex engine (would have taken close to an hour). Switched to flashtext, which uses a trie-based (Aho-Corasick-style) algorithm built specifically for large-scale multi-keyword search - its runtime is independent of how many keywords you're matching. Confirmed: full dataset in about 95 seconds.

**Why gazetteer matching at all for v1:** it's fully interpretable (every match is traceable to an exact term), needs no labeled training data, and is fast to validate by hand. The tradeoff, stated up front: it only finds skills we thought to list, and it can't handle paraphrasing ("builds machine learning models" without the phrase "machine learning" nearby) or new terminology. NER or embedding-based expansion is noted as future work at the end.

No modeling here - this is feature extraction, validated against a manual sample before running on the full dataset.

## Setup

In [1]:
import pandas as pd
from collections import Counter

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE = "./"
postings = pd.read_parquet(f"{BASE}postings_clean.parquet")
print(f"postings: {postings.shape}")

postings: (123849, 32)


## 1. The gazetteer

Organized by category for readability and so we can report skill demand by category later. Each entry is `(display_name, [search terms/aliases])` — most terms have one alias, a few have common variants (e.g. "JavaScript" / "JS").

In [2]:
GAZETTEER = {
    "Programming Languages": {
        "Python": ["python"], "Java": ["java"], "JavaScript": ["javascript", "js"],
        "TypeScript": ["typescript"], "C++": ["c++"], "C#": ["c#"], "Go": ["golang"],
        "Rust": ["rust"], "Ruby": ["ruby"], "PHP": ["php"], "Swift": ["swift"],
        "Kotlin": ["kotlin"], "R": ["r programming"], "MATLAB": ["matlab"],
        "Scala": ["scala"], "SQL": ["sql"], "Bash": ["bash"], "HTML": ["html"], "CSS": ["css"],
    },
    "Data & Machine Learning": {
        "Machine Learning": ["machine learning", "ml"], "Deep Learning": ["deep learning"],
        "Natural Language Processing": ["natural language processing", "nlp"],
        "Computer Vision": ["computer vision"], "Data Analysis": ["data analysis"],
        "Data Visualization": ["data visualization"], "Statistics": ["statistics", "statistical analysis"],
        "TensorFlow": ["tensorflow"], "PyTorch": ["pytorch"], "Scikit-learn": ["scikit-learn", "sklearn"],
        "Pandas": ["pandas"], "NumPy": ["numpy"], "Spark": ["apache spark", "pyspark", "spark"],
        "Hadoop": ["hadoop"], "Tableau": ["tableau"], "Power BI": ["power bi"], "Excel": ["excel"],
        "SAS": ["sas"], "SPSS": ["spss"], "A/B Testing": ["a/b testing"], "ETL": ["etl"],
        "Data Modeling": ["data modeling"], "Data Engineering": ["data engineering"],
        "Predictive Modeling": ["predictive modeling"], "Data Mining": ["data mining"],
    },
    "Cloud & DevOps": {
        "AWS": ["aws", "amazon web services"], "Azure": ["azure"],
        "Google Cloud Platform": ["google cloud platform", "gcp"], "Docker": ["docker"],
        "Kubernetes": ["kubernetes", "k8s"], "CI/CD": ["ci/cd", "continuous integration"],
        "Jenkins": ["jenkins"], "Terraform": ["terraform"], "Ansible": ["ansible"],
        "Linux": ["linux"], "DevOps": ["devops"], "Git": ["git"], "GitHub": ["github"],
        "Microservices": ["microservices"], "REST API": ["rest api", "restful api"],
    },
    "Databases": {
        "SQL Server": ["sql server"], "MySQL": ["mysql"], "PostgreSQL": ["postgresql", "postgres"],
        "MongoDB": ["mongodb"], "Oracle": ["oracle database", "oracle db"], "NoSQL": ["nosql"],
        "Redis": ["redis"], "Snowflake": ["snowflake"], "BigQuery": ["bigquery"],
    },
    "Software Engineering": {
        "React": ["react.js", "reactjs", "react"], "Angular": ["angular"], "Vue.js": ["vue.js", "vuejs"],
        "Node.js": ["node.js", "nodejs"], "Django": ["django"], "Flask": ["flask"],
        "Spring": ["spring boot", "spring framework"], ".NET": [".net"],
        "Agile": ["agile"], "Scrum": ["scrum"], "Object-Oriented Programming": ["object-oriented", "oop"],
        "System Design": ["system design"], "QA/Testing": ["quality assurance", "software testing"],
    },
    "Business & Soft Skills": {
        "Project Management": ["project management"], "Communication": ["communication skills"],
        "Leadership": ["leadership"], "Problem Solving": ["problem solving", "problem-solving"],
        "Critical Thinking": ["critical thinking"], "Customer Service": ["customer service"],
        "Sales": ["sales"], "Negotiation": ["negotiation"], "Time Management": ["time management"],
        "Public Speaking": ["public speaking"], "Strategic Planning": ["strategic planning"],
        "Budgeting": ["budgeting"], "Financial Analysis": ["financial analysis"],
        "Marketing": ["marketing"], "Digital Marketing": ["digital marketing"], "SEO": ["seo"],
        "Social Media Marketing": ["social media marketing"], "Business Development": ["business development"],
        "Account Management": ["account management"], "Human Resources": ["human resources", "hr"],
        "Recruiting": ["recruiting", "recruitment"], "Training": ["training and development"],
        "Consulting": ["consulting"],
    },
    "Finance & Accounting": {
        "Accounting": ["accounting"], "Bookkeeping": ["bookkeeping"], "GAAP": ["gaap"],
        "Financial Reporting": ["financial reporting"], "Auditing": ["auditing"],
        "Tax Preparation": ["tax preparation"], "QuickBooks": ["quickbooks"],
        "Financial Modeling": ["financial modeling"], "Risk Management": ["risk management"],
    },
    "Healthcare": {
        "Nursing": ["nursing"], "Patient Care": ["patient care"], "Clinical Research": ["clinical research"],
        "Medical Coding": ["medical coding"], "HIPAA": ["hipaa"],
        "Electronic Health Records": ["electronic health records", "ehr"], "CPR": ["cpr"],
    },
    "Design": {
        "Adobe Photoshop": ["photoshop"], "Adobe Illustrator": ["illustrator"],
        "UX Design": ["ux design", "user experience design"], "UI Design": ["ui design"],
        "Graphic Design": ["graphic design"], "Figma": ["figma"],
    },
    "Manufacturing & Supply Chain": {
        "AutoCAD": ["autocad"], "SolidWorks": ["solidworks"], "Six Sigma": ["six sigma"],
        "Lean Manufacturing": ["lean manufacturing"], "Quality Control": ["quality control"],
        "Supply Chain Management": ["supply chain"], "Logistics": ["logistics"],
    },
}

n_skills = sum(len(cat) for cat in GAZETTEER.values())
n_aliases = sum(len(aliases) for cat in GAZETTEER.values() for aliases in cat.values())
print(f"{len(GAZETTEER)} categories, {n_skills} skills, {n_aliases} total aliases")

10 categories, 133 skills, 159 total aliases


## 2. Build the matcher

Using flashtext's KeywordProcessor - a trie built once from all aliases, then a single pass per document finds every match regardless of how many keywords are loaded. Case-insensitive matching, word-boundary-aware by default.

In [3]:
from flashtext import KeywordProcessor

skill_category = {}
kp = KeywordProcessor(case_sensitive=False)

for category, skills in GAZETTEER.items():
    for display_name, aliases in skills.items():
        skill_category[display_name] = category
        for alias in aliases:
            kp.add_keyword(alias, display_name)

def match_skills(text) -> set:
    if not isinstance(text, str):
        return set()
    return set(kp.extract_keywords(text))

print(f"trie built from {n_aliases} aliases covering {n_skills} skills")

trie built from 159 aliases covering 133 skills


## 3. Validate on a manual sample before running on everything

Pulling 5 postings and hand-checking the matches make sense before committing to running this across all 123,849 rows.

In [4]:
sample = postings.sample(5, random_state=42)

for _, row in sample.iterrows():
    matches = sorted(match_skills(row["description"]))
    print(f"--- {row['title']} ---")
    print(f"matched: {matches if matches else '(none)'}")
    print()

--- Senior Automation Engineer - Power Systems ---
matched: ['AutoCAD', 'Communication', 'Excel', 'System Design']

--- DISH Installation Technician - Field ---
matched: ['Leadership']

--- Order Builder ---
matched: (none)

--- Mountain Multimedia Journalist, KMGH ---
matched: ['Leadership']

--- Licensed Practical Nurse (LPN) ---
matched: ['Nursing']



**Check this output before continuing.** If matches look wrong (false positives, obvious misses), fix the gazetteer before running on the full dataset — cheap to fix now, expensive to redo after.

## 4. Run on the full dataset

In [5]:
postings["extracted_skills"] = postings["description"].apply(lambda t: sorted(match_skills(t)))

n_with_skills = (postings["extracted_skills"].str.len() > 0).sum()
avg_skills = postings["extracted_skills"].str.len().mean()
print(f"postings with >=1 extracted skill: {n_with_skills:,} / {len(postings):,} ({n_with_skills/len(postings)*100:.1f}%)")
print(f"average skills per posting (including 0-skill postings): {avg_skills:.2f}")

postings with >=1 extracted skill: 109,686 / 123,849 (88.6%)
average skills per posting (including 0-skill postings): 3.09


## 5. Coverage and top skills

In [6]:
all_matches = [skill for skills in postings["extracted_skills"] for skill in skills]
top_skills = Counter(all_matches).most_common(20)

top_skills_df = pd.DataFrame(top_skills, columns=["skill", "postings_mentioning"])
top_skills_df["category"] = top_skills_df["skill"].map(skill_category)
top_skills_df

,skill,postings_mentioning,category
0,Leadership,29355,Business & Soft Skills
1,Communication,29034,Business & Soft Skills
2,Sales,28777,Business & Soft Skills
3,Customer Service,24985,Business & Soft Skills
4,Problem Solving,19182,Business & Soft Skills
5,Excel,18107,Data & Machine Learning
6,Marketing,15257,Business & Soft Skills
7,Recruiting,15159,Business & Soft Skills
8,Human Resources,14288,Business & Soft Skills
9,Project Management,10229,Business & Soft Skills


## 6. Save extracted skills (long format)

One row per `(job_id, skill)` pair — mirrors the shape of `jobs/job_skills.csv` so it's easy to compare coverage against the dataset's own (coarser) skill field.

In [7]:
extracted_long = postings[["job_id", "extracted_skills"]].explode("extracted_skills").dropna()
extracted_long = extracted_long.rename(columns={"extracted_skills": "skill"})
extracted_long["category"] = extracted_long["skill"].map(skill_category)

extracted_long.to_parquet(f"{BASE}job_skills_extracted.parquet", index=False)
print(f"saved job_skills_extracted.parquet: {extracted_long.shape[0]:,} (job_id, skill) pairs")
extracted_long.head()

saved job_skills_extracted.parquet: 383,288 (job_id, skill) pairs


,job_id,skill,category
0,921716,Adobe Illustrator,Design
0,921716,Adobe Photoshop,Design
0,921716,Graphic Design,Design
0,921716,Marketing,Business & Soft Skills
0,921716,Sales,Business & Soft Skills


## 7. Known limitations and next steps

- **Coverage ceiling:** whatever fraction of postings show `0` extracted skills either genuinely don't mention any gazetteer term, or use phrasing/terminology not in our list yet. Worth spot-checking a sample of 0-skill postings to tell which.
- **No synonym/abbreviation completeness:** the gazetteer has some aliases per skill but isn't exhaustive (e.g. doesn't yet catch every way "JavaScript" might be written).
- **No context awareness:** a posting mentioning "not required: Python experience" would still match "Python" — the matcher doesn't understand negation.
- **Next steps:** expand the gazetteer using a real external taxonomy (e.g. O*NET Technology Skills, ESCO) instead of hand-curated terms; consider layering an embedding-based or NER pass to catch paraphrased mentions; use this v1 output as a labeled-ish baseline to evaluate any v2 approach against.